In [ ]:
from pathlib import Path
import json
import runpy
import pandas as pd
from IPython.display import display, Markdown
import resources

REPO = Path(resources.__file__).resolve().parents[1]

# Task

Joint UT/FT geometry → displacement fields → full 201-point curves. This notebook configures data, HPO and training; inspect saved predictions in **ML-DualPostProcessing.ipynb**. Feature construction and worked calculations belong in `../samples/`.

## ML Model Name: Two-stage dual Transformer

### DATA

Select a saved Trial 1 for configuration review, or explicitly supply the local parent containing `MLdata` when training. Saved artifacts do not contain the full training dataset. No HPC connection or training occurs by default.

In [ ]:
RUN_ROOT = REPO / "data"
REFERENCE_RUN = RUN_ROOT / "MULTI/Dual/Transformer/dual-MULTI-trial1"
DATA_ROOT = None
NSIMS = "all"
SEED = 42
RUN_TRAINING = False
RUN_HPO = False
ALLOW_CPU = False

reference = json.loads((REFERENCE_RUN / "results/metrics.json").read_text())
display(pd.DataFrame(reference["split_sizes"].items(), columns=["Split", "Paired specimens"]))
display(Markdown(f"Saved evaluation: **{reference['evaluation_split']}**. Best epoch: **{reference['best_epoch']}**. "
                 "The 881 test samples remain outside HPO selection."))

### HPO

The HPC entry point owns the search space and resumable study. Review the adjacent `HPC/DualOutputs/README.md` before launching. This cell only displays the plan unless `RUN_HPO=True`; do not run the production study on this laptop. Fixed validation ranking is independent of tuned training-loss weights.

In [ ]:
HPO_SCRIPT = REPO / "p2-DisorderML/HPC/DualOutputs/A0-HPC-Dual-HPO.py"
STUDY_NAME = "dual-joint-hpo1"
TARGET_TRIALS = 200
TIMEOUT_HOURS = 230
HPO_ARGS = ["--study-name", STUDY_NAME, "--target-trials", str(TARGET_TRIALS),
            "--timeout-hours", str(TIMEOUT_HOURS), "--run-root", str(RUN_ROOT)]
print("HPC: sbatch -J dual-joint-hpo1 B1_ML-new.sh DualOutputs/A0-HPC-Dual-HPO.py")
print("Local HPO is disabled:", not RUN_HPO)
if RUN_HPO:
    if DATA_ROOT is None:
        raise ValueError("Set DATA_ROOT to a local MLdata parent before HPO.")
    runpy.run_path(str(HPO_SCRIPT))["main"](HPO_ARGS + ["--data-path", str(DATA_ROOT)])

### MODEL

The preset reproduces Trial 1's documented architecture and optimiser settings. One encoder per stage is shared across UT/FT; one optimiser trains all four outputs jointly. Training is opt-in, uses a new run label, and saves results automatically. Loss monitoring and all detailed prediction diagnostics belong in the post-processing notebook.

In [ ]:
RUN_LABEL = "dual-local-trial"
MODEL_ARGS = [
    "--data-path", str(DATA_ROOT), "--run-root", str(RUN_ROOT), "--run-label", RUN_LABEL,
    "--nsims", str(NSIMS), "--seed", str(SEED), "--eval-split", "val",
]
if ALLOW_CPU:
    MODEL_ARGS += ["--allow-cpu"]
config = reference["run_config"]
display(pd.DataFrame([(k, v) for k, v in config.items()
                     if k.startswith(("field_", "curve_")) or k in ("lr", "batch", "epochs", "loss", "weight_decay")],
                    columns=["Trial 1 setting", "Value"]))
if RUN_TRAINING:
    if DATA_ROOT is None:
        raise ValueError("Set DATA_ROOT before training; saved results are not the training dataset.")
    MODEL = runpy.run_path(str(REPO / "p2-DisorderML/HPC/DualOutputs/A0-HPC-Dual-trial1.py"))["main"](MODEL_ARGS)
else:
    print("Configuration review only: no data loading, training or checkpoint overwrite.")